# Vintern-1B-v2 (5CD-AI): nhận dạng chữ viết tay tiếng Việt trên ảnh trong dataset Kaggle

Notebook đọc **mọi ảnh** trong `/kaggle/input`, chạy `5CD-AI/Vintern-1B-v2` theo batch, lưu kết quả vào `/kaggle/working/vintern_out/`.

Đây là bước **chạy thử không tinh chỉnh (zero-shot)**, dùng để:
1. đo mức cơ sở (CER theo ký tự và âm tiết) khi bạn có vài chục dòng đáp án;
2. tạo **bản nháp** `labeling.csv` để bạn sửa tay thành dữ liệu huấn luyện cho bước tinh chỉnh sau.

**Trước khi chạy**
- Accelerator: một GPU T4 hoặc P100 là đủ (mô hình ~1B tham số, ~2 GB).
- Settings → **Internet: On** để tải model. Hoặc Add Input một bản Vintern-1B-v2 rồi đặt `MODEL_ID` là đường dẫn `/kaggle/input/...`.
- Add dataset chứa ảnh qua **Add Input**. Ảnh nên là **từng dòng đã cắt riêng**.

In [ ]:
import importlib, importlib.metadata as md, subprocess, sys

# Ô khởi động: đảm bảo dùng transformers 4.x (mã nguồn tuỳ biến của Vintern chưa chạy được với 5.x).
# Chạy lại bao nhiêu lần cũng được: chỉ cài khi bản trên đĩa chưa đúng. Luôn chạy ô này đầu tiên, cả sau khi Restart.
PIN = "4.51.3"      # cài được trên Python 3.13 của Kaggle


def disk_version():
    importlib.invalidate_caches()
    try:
        return md.version("transformers")
    except md.PackageNotFoundError:
        return None


dv = disk_version()
if dv is None or int(dv.split(".")[0]) >= 5:
    print(f"transformers trên đĩa: {dv} → đang cài {PIN} ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"transformers=={PIN}"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "accelerate", "einops", "timm", "sentencepiece"], check=True)

dv = disk_version()
loaded = sys.modules.get("transformers")
if loaded is not None and int(loaded.__version__.split(".")[0]) >= 5:
    print(f"⚠ Trên đĩa đã là transformers {dv}, nhưng kernel đang giữ bản {loaded.__version__} trong bộ nhớ.\n"
          "→ Khởi động lại kernel (Restart), rồi chạy lại từ ô này (ô sẽ bỏ qua bước cài).")
else:
    print("OK: transformers", dv, "- có thể chạy tiếp các ô sau.")

In [ ]:
import os, re, json, time, unicodedata
from pathlib import Path

# ======================= CẤU HÌNH =======================
INPUT_ROOT     = "/kaggle/input"          # nơi Kaggle mount dataset
DATASET_SUBDIR = ""                       # vd "ten-dataset"; để trống = quét toàn bộ /kaggle/input
MODEL_ID       = "5CD-AI/Vintern-1B-v2"   # hoặc đường dẫn local trong /kaggle/input/...

PROMPT_KEY     = "ocr_line"               # xem PROMPTS bên dưới
DTYPE          = "auto"                   # "auto" = bf16 nếu GPU hỗ trợ (A100...), ngược lại fp16 (T4/P100)
BATCH_SIZE     = 8                        # hết VRAM sẽ tự chia đôi batch
MAX_TILES      = 6                        # số mảnh 448x448 tối đa cho mỗi ảnh. Nhiều = rõ hơn nhưng chậm hơn
PAD_THIN_LINES = True                     # ảnh dòng rất dài/mỏng: đệm nền thay vì kéo méo tỉ lệ
USE_THUMBNAIL  = True                     # thêm 1 ảnh thu nhỏ toàn cảnh vào mỗi ảnh có nhiều mảnh
ISOLATE_RED    = False                    # True: chỉ giữ mực đỏ (bỏ dòng kẻ ô, mực đen) trước khi đọc
ONE_LINE       = True                     # True: gộp xuống dòng thành 1 dòng (hợp với ảnh 1 dòng chữ)

MAX_NEW_TOKENS = 256
NUM_BEAMS      = 1                        # card mẫu dùng 3 (chậm hơn nhiều); với chép chữ nên để 1
REP_PENALTY    = 1.0                      # card mẫu dùng 2.5 (cho mô tả ảnh) → làm hỏng chép chữ vì ép tránh lặp từ

LIMIT          = None                     # vd 16 để chạy thử vài ảnh đầu; None = tất cả
GT_CSV         = None                     # (tuỳ chọn) CSV đáp án có cột file,text để đo CER ở cuối

OUT_DIR = Path("/kaggle/working/vintern_out")
OUT_DIR.mkdir(parents=True, exist_ok=True)
RESULT_JSONL = OUT_DIR / "results.jsonl"  # ghi từng ảnh một, chạy lại sẽ tự bỏ qua ảnh đã xong

PROMPTS = {
    "ocr_line": (
        "<image>\nChép lại chính xác dòng chữ viết tay tiếng Việt trong ảnh, "
        "giữ đầy đủ dấu thanh và dấu mũ. Không sửa chính tả, không giải thích. Chỉ trả về văn bản."
    ),
    "ocr_page": (
        "<image>\nChép lại toàn bộ chữ trong ảnh theo thứ tự đọc, mỗi dòng một dòng. "
        "Giữ đầy đủ dấu thanh và dấu mũ. Chỉ trả về văn bản."
    ),
    "markdown": "<image>\nTrích xuất thông tin chính trong ảnh và trả về dạng markdown.",   # prompt gợi ý trong card
}

## 1. Liệt kê ảnh trong dataset

In [ ]:
EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
root = Path(INPUT_ROOT) / DATASET_SUBDIR

def natural_key(p):
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r"(\d+)", str(p))]

image_paths = sorted((p for p in root.rglob("*") if p.suffix.lower() in EXTS), key=natural_key)
if LIMIT:
    image_paths = image_paths[:LIMIT]

print(f"Tìm thấy {len(image_paths)} ảnh trong {root}")
for p in image_paths[:5]:
    print("  ", p)
assert image_paths, "Không có ảnh nào. Kiểm tra INPUT_ROOT/DATASET_SUBDIR và đã Add dataset chưa."

## 2. Nạp model

In [ ]:
import torch
import transformers
from transformers import AutoModel, AutoTokenizer

assert int(transformers.__version__.split(".")[0]) < 5, (
    f"Kernel đang giữ transformers {transformers.__version__}, quá mới so với mã nguồn của Vintern "
    "(lỗi 'all_tied_weights_keys'). Chạy ô khởi động (ô code đầu tiên, nó tự cài 4.51.3 nếu thiếu), "
    "khởi động lại kernel, rồi chạy lại từ ô khởi động."
)
print("transformers", transformers.__version__)
assert torch.cuda.is_available(), "Cần bật GPU: Settings → Accelerator."
DEVICE = "cuda:0"
cap = torch.cuda.get_device_capability(0)
if DTYPE == "auto":
    DTYPE_T = torch.bfloat16 if cap[0] >= 8 else torch.float16     # T4/P100 không có bf16 gốc
else:
    DTYPE_T = getattr(torch, DTYPE)
print("GPU:", torch.cuda.get_device_name(0), "| dtype:", DTYPE_T)

model = AutoModel.from_pretrained(
    MODEL_ID, torch_dtype=DTYPE_T, low_cpu_mem_usage=True, trust_remote_code=True,
).eval().to(DEVICE)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True, use_fast=False)

print("Đã nạp model. VRAM dùng (GB):", round(torch.cuda.memory_allocated(0) / 1e9, 2))
print("Có batch_chat:", hasattr(model, "batch_chat"))

## 3. Hàm tiện ích: tiền xử lý ảnh, suy luận theo batch

In [ ]:
import math
import numpy as np
from PIL import Image, ImageOps
import torchvision.transforms as T
from torchvision.transforms.functional import InterpolationMode

IMAGENET_MEAN, IMAGENET_STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
TILE = 448
transform = T.Compose([
    T.Resize((TILE, TILE), interpolation=InterpolationMode.BICUBIC),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


def load_image(path):
    img = ImageOps.exif_transpose(Image.open(path)).convert("RGB")
    if ISOLATE_RED:
        a = np.asarray(img).astype(np.int16)
        r, g, b = a[..., 0], a[..., 1], a[..., 2]
        strength = np.clip((r - np.maximum(g, b)) * 4, 0, 255)      # độ "đỏ" của từng điểm ảnh
        gray = (255 - strength).astype(np.uint8)
        img = Image.fromarray(np.stack([gray] * 3, axis=-1))
    return img


def pad_thin(img, max_ratio):
    """Ảnh dài/mỏng hơn max_ratio:1 thì đệm nền (màu giấy) thêm chiều cao, thay vì bị kéo méo khi chia mảnh."""
    w, h = img.size
    if w / h <= max_ratio:
        return img
    new_h = math.ceil(w / max_ratio)
    bg = tuple(int(v) for v in np.median(np.asarray(img).reshape(-1, 3), axis=0))   # màu giấy
    canvas = Image.new("RGB", (w, new_h), bg)
    canvas.paste(img, (0, (new_h - h) // 2))
    return canvas


def find_closest_aspect_ratio(aspect_ratio, target_ratios, width, height, image_size):
    best_ratio_diff, best_ratio = float("inf"), (1, 1)
    area = width * height
    for ratio in target_ratios:
        diff = abs(aspect_ratio - ratio[0] / ratio[1])
        if diff < best_ratio_diff:
            best_ratio_diff, best_ratio = diff, ratio
        elif diff == best_ratio_diff and area > 0.5 * image_size * image_size * ratio[0] * ratio[1]:
            best_ratio = ratio
    return best_ratio


def dynamic_preprocess(image, min_num=1, max_num=12, image_size=448, use_thumbnail=False):
    """Chia ảnh thành lưới các mảnh 448x448 (cách làm chuẩn của họ InternVL / Vintern)."""
    ow, oh = image.size
    ratios = sorted(
        {(i, j) for n in range(min_num, max_num + 1) for i in range(1, n + 1) for j in range(1, n + 1)
         if min_num <= i * j <= max_num},
        key=lambda x: x[0] * x[1],
    )
    gx, gy = find_closest_aspect_ratio(ow / oh, ratios, ow, oh, image_size)
    resized = image.resize((image_size * gx, image_size * gy))
    tiles = []
    for k in range(gx * gy):
        x0, y0 = (k % gx) * image_size, (k // gx) * image_size
        tiles.append(resized.crop((x0, y0, x0 + image_size, y0 + image_size)))
    if use_thumbnail and len(tiles) != 1:
        tiles.append(image.resize((image_size, image_size)))
    return tiles


def preprocess(img):
    if PAD_THIN_LINES:
        img = pad_thin(img, MAX_TILES)
    tiles = dynamic_preprocess(img, max_num=MAX_TILES, image_size=TILE, use_thumbnail=USE_THUMBNAIL)
    return torch.stack([transform(t) for t in tiles])               # [n_tiles, 3, 448, 448]


def clean_text(s):
    s = re.sub(r"^```[a-zA-Z]*|```$", "", s.strip(), flags=re.M).strip()
    if ONE_LINE:
        s = re.sub(r"\s*\n\s*", " ", s)
    return unicodedata.normalize("NFC", re.sub(r"[ \t]+", " ", s)).strip()


GEN = dict(max_new_tokens=MAX_NEW_TOKENS, do_sample=False, num_beams=NUM_BEAMS, repetition_penalty=REP_PENALTY)


@torch.inference_mode()
def infer_batch(imgs, prompt):
    """Chạy một batch ảnh với cùng một prompt. Trả về list[(văn bản, số mảnh)]."""
    per_img = [preprocess(im) for im in imgs]
    counts = [t.shape[0] for t in per_img]
    pixel_values = torch.cat(per_img).to(DEVICE, DTYPE_T)
    questions = [prompt] * len(imgs)
    if hasattr(model, "batch_chat"):
        outs = model.batch_chat(tokenizer, pixel_values, num_patches_list=counts,
                                questions=questions, generation_config=dict(GEN))
    else:                                                           # dự phòng: chạy lần lượt từng ảnh
        outs, off = [], 0
        for c, q in zip(counts, questions):
            r = model.chat(tokenizer, pixel_values[off:off + c], q, dict(GEN))
            outs.append(r[0] if isinstance(r, tuple) else r)
            off += c
    return [(clean_text(o), c) for o, c in zip(outs, counts)]


def run_batch(imgs, prompt):
    """Chạy batch; nếu hết VRAM thì tự chia đôi batch cho tới khi chạy được."""
    oom = False
    try:
        return infer_batch(imgs, prompt)
    except torch.cuda.OutOfMemoryError:
        oom = True
    torch.cuda.empty_cache()
    if len(imgs) == 1:
        raise RuntimeError("CUDA OOM ngay cả với 1 ảnh (giảm MAX_TILES)")
    mid = len(imgs) // 2
    return run_batch(imgs[:mid], prompt) + run_batch(imgs[mid:], prompt)

## 4. Thử nhanh trên vài ảnh đầu

In [ ]:
prompt = PROMPTS[PROMPT_KEY]
test_paths = image_paths[:4]
t0 = time.time()
outs = run_batch([load_image(p) for p in test_paths], prompt)
print(f"{len(test_paths)} ảnh trong {time.time() - t0:.1f}s")
for p, (txt, n) in zip(test_paths, outs):
    print(f"{p.name}  [{n} mảnh]  →  {txt}")

## 5. Chạy trên toàn bộ ảnh (tự lưu từng ảnh, chạy lại sẽ tiếp tục)

In [ ]:
from tqdm.auto import tqdm

done = set()
if RESULT_JSONL.exists():
    with open(RESULT_JSONL, encoding="utf-8") as f:
        for line in f:
            try:
                done.add(json.loads(line)["file"])
            except Exception:
                pass
print(f"Đã có {len(done)} ảnh xong từ lần chạy trước.")

todo = [p for p in image_paths if str(p.relative_to(INPUT_ROOT)) not in done]


def area(p):
    with Image.open(p) as im:
        return im.size[0] * im.size[1]


todo.sort(key=area)       # ảnh cùng cỡ đứng cạnh nhau → batch đều hơn, nhanh hơn
print(f"Còn {len(todo)} ảnh cần xử lý. BATCH_SIZE={BATCH_SIZE}, MAX_TILES={MAX_TILES}")

with open(RESULT_JSONL, "a", encoding="utf-8") as fout:
    def write(rec):
        fout.write(json.dumps(rec, ensure_ascii=False) + "\n")
        fout.flush()

    for start in tqdm(range(0, len(todo), BATCH_SIZE)):
        rels, imgs = [], []
        for path in todo[start:start + BATCH_SIZE]:
            rel = str(path.relative_to(INPUT_ROOT))
            try:
                imgs.append(load_image(path))
                rels.append(rel)
            except Exception as e:
                write({"file": rel, "text": "", "error": f"{type(e).__name__}: {e}"})
        if not imgs:
            continue

        t0 = time.time()
        try:
            outs = run_batch(imgs, prompt)
        except Exception as e:
            for rel in rels:
                write({"file": rel, "text": "", "error": f"{type(e).__name__}: {e}"})
            torch.cuda.empty_cache()
            continue
        sec = round((time.time() - t0) / len(imgs), 3)               # thời gian trung bình mỗi ảnh trong batch

        for rel, img, (txt, n_tiles) in zip(rels, imgs, outs):
            write({"file": rel, "text": txt, "width": img.size[0], "height": img.size[1],
                   "tiles": n_tiles, "sec": sec})
        if (start // BATCH_SIZE) % 10 == 0:
            torch.cuda.empty_cache()
print("Xong.")

## 6. Xuất kết quả và file nháp để sửa nhãn

In [ ]:
import pandas as pd

rows = [json.loads(l) for l in open(RESULT_JSONL, encoding="utf-8")]
rows.sort(key=lambda r: natural_key(r["file"]))          # trả về thứ tự tự nhiên (đã xử lý theo cỡ ảnh)
df = pd.DataFrame(rows)
cols = [c for c in ["file", "text", "error", "sec", "tiles", "width", "height"] if c in df.columns]
df[cols].to_csv(OUT_DIR / "results.csv", index=False, encoding="utf-8-sig")   # utf-8-sig để Excel mở đúng tiếng Việt

# File nháp để sửa tay: sửa cột "text" cho đúng, đánh dấu "checked"=1 khi đã duyệt xong dòng đó
draft = df[["file", "text"]].copy()
draft["checked"] = ""
draft.to_csv(OUT_DIR / "labeling.csv", index=False, encoding="utf-8-sig")

print("Đã lưu:", OUT_DIR / "results.csv", "và", OUT_DIR / "labeling.csv", "| số dòng:", len(df))
if "error" in df.columns:
    print("Ảnh lỗi:", int(df["error"].notna().sum()))
if "sec" in df.columns:
    print(f"Thời gian trung vị: {df['sec'].median():.3f}s/ảnh  →  khoảng {1 / max(df['sec'].median(), 1e-9):.1f} ảnh/giây")
df[cols].head(20)

## 7. Xem nhanh vài ảnh cùng kết quả

In [ ]:
import matplotlib.pyplot as plt

sample = [r for r in rows if r.get("text")][:6]
fig, axes = plt.subplots(len(sample), 1, figsize=(10, 2.6 * len(sample)))
axes = np.atleast_1d(axes)
for ax, r in zip(axes, sample):
    ax.imshow(load_image(Path(INPUT_ROOT) / r["file"]))
    ax.set_title(r["text"][:120], fontsize=11, loc="left")
    ax.axis("off")
plt.tight_layout()
plt.show()

## 8. (Tuỳ chọn) Đo CER / tỉ lệ sai âm tiết so với đáp án

Chuẩn bị CSV có cột `file` (đường dẫn tương đối như trong `results.csv`, hoặc chỉ tên file) và cột `text` (đáp án).
Đặt `GT_CSV` ở ô cấu hình rồi chạy lại ô này. Bạn có thể dùng chính `labeling.csv` sau khi đã sửa tay (Add làm dataset).

In [ ]:
def lev(a, b):
    prev = list(range(len(b) + 1))
    for i, x in enumerate(a, 1):
        cur = [i]
        for j, y in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (x != y)))
        prev = cur
    return prev[-1]


def norm(s, lower=True, strip_punct=True):
    s = unicodedata.normalize("NFC", str(s))
    if lower:
        s = s.lower()
    if strip_punct:
        s = re.sub(r"[^\w\s]", " ", s)
    return re.sub(r"\s+", " ", s).strip()


if GT_CSV and Path(GT_CSV).exists():
    gt = pd.read_csv(GT_CSV, dtype=str).fillna("")
    gt_map = {r.file: r.text for r in gt.itertuples()}
    gt_base = {Path(k).name: v for k, v in gt_map.items()}
    ce = cn = we = wn = n = 0
    worst = []
    for r in rows:
        ref = gt_map.get(r["file"], gt_base.get(Path(r["file"]).name))
        if ref is None or not ref.strip():
            continue
        h, g = norm(r.get("text", "")), norm(ref)
        e_c, e_w = lev(h, g), lev(h.split(), g.split())
        ce += e_c; cn += len(g); we += e_w; wn += len(g.split()); n += 1
        worst.append((e_c, r["file"], h, g))
    print(f"So với {n} dòng đáp án:  CER = {ce / max(cn, 1):.2%}   |   tỉ lệ sai âm tiết = {we / max(wn, 1):.2%}")
    for e_c, f, h, g in sorted(worst, reverse=True)[:8]:
        print(f"\n[{e_c} lỗi] {f}\n  máy : {h}\n  đúng: {g}")
else:
    print("Chưa đặt GT_CSV, bỏ qua bước đo.")

## Ghi chú
- **Tốc độ**: thời gian chủ yếu phụ thuộc `MAX_TILES` (mỗi mảnh ≈ 256 token ảnh) và `BATCH_SIZE`. Với dòng chữ ngắn thử `MAX_TILES = 4`; chữ nhỏ/nét mảnh thì tăng lên 8. Cột `sec` trong CSV giúp so các cấu hình.
- **Hết VRAM**: notebook tự chia đôi batch; nếu vẫn hết, giảm `MAX_TILES`.
- **Chữ sai dấu nhiều**: tăng `MAX_TILES`, bật `ISOLATE_RED` nếu mực đỏ lẫn đường kẻ ô, và thử `NUM_BEAMS = 3` (chậm hơn).
- **Lỗi `'InternVLChatModel' object has no attribute 'all_tied_weights_keys'`**: do `transformers` 5.x. Ô khởi động ở đầu notebook tự cài `transformers==4.51.3` khi cần. Nếu nó báo kernel còn giữ bản 5.x thì khởi động lại kernel rồi chạy lại **từ chính ô khởi động** (đừng bỏ qua ô này, vì sau khi khởi động lại phiên gói đã cài có thể bị mất).
- **Lỗi tương thích khác khi nạp model hoặc gọi `batch_chat`**: thử các bản 4.x khác (vd `4.48.3` hoặc `4.49.0`); đừng dùng `4.44.2` vì cần `tokenizers` 0.19 không có gói cho Python 3.13. Notebook tự chuyển sang chạy từng ảnh nếu mô hình không có `batch_chat`.
- **Bước tiếp theo**: sửa tay `labeling.csv` (vài trăm dòng) rồi tinh chỉnh Vintern-1B-v2 trên đó.